# Capstone Function 3
You're working on a drug discovery project, testing combinations of three compounds to create a new medicine.\
Each experiment is stored in ```initial_inputs.npy``` as a 3D array, where each row lists the amounts of the three compounds used.  After each experiment, you record the number of adverse reactions, stored in ```initial_outputs.npy``` as a 1D array.\
Your goal is to minimise the side effects, it is framed as a maximisation by optimising a transformed output (e.g. the negative of side effects)

 Input | Output | Goal |
|-------|--------|------|
| 3D Array (15, 3) | 1D Array (15, ) | Maximise |

## Initial Submission

This section contains the initial Bayesian Optimization implementation for the 3D drug discovery problem (minimizing adverse reactions).

### Step 1: Import Required Libraries

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from botorch.acquisition import ExpectedImprovement
from botorch.optim import optimize_acqf
from gpytorch.mlls import ExactMarginalLogLikelihood
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")

### Step 2: Load and Display Initial Data

Load the initial input and output data for the drug discovery problem (3 compound combinations).

In [ ]:
# Load initial data
X_init = np.load('../../data/f3/updated_inputs - Week 4.npy')
y_init = np.load('../../data/f3/updated_outputs - Week 4.npy')

# Display data characteristics
print("Initial Data Summary:")
print(f"Input shape: {X_init.shape} (15 experiments, 3 compounds)")
print(f"Output shape: {y_init.shape}")
print(f"\nInput range: [{X_init.min():.4f}, {X_init.max():.4f}]")
print(f"Output range: [{y_init.min():.6f}, {y_init.max():.6f}]")
print(f"Output mean: {y_init.mean():.6f}")
print(f"Output std: {y_init.std():.6f}")
print(f"\nBest observed value: {y_init.max():.6f}")
print(f"Best input location: {X_init[y_init.argmax()]}")

# Display first few samples
print(f"\nFirst 5 samples:")
for i in range(min(5, len(X_init))):
    print(f"  X[{i}] = {X_init[i]}, y[{i}] = {y_init[i]:.6f}")

In [ ]:
# Visualize initial data in 3D space
fig = plt.figure(figsize=(14, 5))

# 3D scatter plot
ax1 = fig.add_subplot(121, projection='3d')
scatter = ax1.scatter(X_init[:, 0], X_init[:, 1], X_init[:, 2], c=y_init, s=150, cmap='viridis', edgecolors='black', linewidth=1.5)
best_idx = y_init.argmax()
ax1.scatter(X_init[best_idx, 0], X_init[best_idx, 1], X_init[best_idx, 2], s=400, c='red', marker='*', edgecolors='black', linewidth=2)
ax1.set_xlabel('Compound 1', fontsize=10)
ax1.set_ylabel('Compound 2', fontsize =10)
ax1.set_zlabel('Compound 3', fontsize=10)
ax1.set_title('3D Input Space', fontsize=12, fontweight='bold')
fig.colorbar(scatter, ax=ax1, label='Output Value', shrink=0.6)

# Pair plot (2D projections)
ax2 = fig.add_subplot(122)
scatter2 = ax2.scatter(X_init[:, 0], X_init[:, 1], c=y_init, s=150, cmap='viridis', edgecolors='black', linewidth=1.5)
ax2.scatter(X_init[best_idx, 0], X_init[best_idx, 1], s=400, c='red', marker='*', edgecolors='black', linewidth=2, label='Best')
ax2.set_xlabel('Compound 1', fontsize=10)
ax2.set_ylabel('Compound 2', fontsize=10)
ax2.set_title('2D Projection (Comp 1 vs 2)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()
plt.colorbar(scatter2, ax=ax2, label='Output Value')

plt.tight_layout()
plt.show()

### Step 3: Define Hyperparameters

**Hyperparameter Choices for 3D Drug Discovery:**

1. **Gaussian Process Kernel**: Matern 5/2  
   - Suitable for smooth drug response surfaces
   - Robust to local variations in efficacy

2. **Acquisition Function**: Expected Improvement
   - Balances exploring new compound combinations with exploiting known effective ones
   - 15 initial samples provides reasonable coverage of 3D space

3. **Restarts & Raw Samples**: 15 restarts, 1024 raw samples
   - Higher than 2D problems due to increased dimensionality
   - Ensures thorough search of acquisition landscape

4. **Input Bounds**: [0, 1.0] for all dimensions
   - Required by submission format - all inputs must be in range [0, 1.0]

In [ ]:
# Define hyperparameters
# All inputs must be in range [0, 1.0] per submission requirements
N_DIM = X_init.shape[1]  # Number of dimensions (3)
BOUNDS = torch.tensor([[0.0] * N_DIM, [1.0] * N_DIM], dtype=torch.float64)

NUM_RESTARTS = 15
RAW_SAMPLES = 1024

print("Hyperparameters:")
print(f"  Input dimensions: {N_DIM}")
print(f"  Input bounds: [0, 1.0] for all {N_DIM} dimensions")
print(f"  Acquisition function: Expected Improvement (EI)")
print(f"  GP Kernel: Matern 5/2")
print(f"  Number of restarts: {NUM_RESTARTS}")
print(f"  Raw samples: {RAW_SAMPLES}")

### Step 4: Build Gaussian Process Surrogate Model

In [ ]:
# Convert data to PyTorch tensors
X_train = torch.tensor(X_init, dtype=torch.float64)
y_train = torch.tensor(y_init, dtype=torch.float64).unsqueeze(-1)

print(f"Training data shape: X={X_train.shape}, y={y_train.shape}")

# Create and train Gaussian Process model
gp_model = SingleTaskGP(X_train, y_train)
mll = ExactMarginalLogLikelihood(gp_model.likelihood, gp_model)

print("\nTraining Gaussian Process model...")
fit_gpytorch_mll(mll)
print("✓ Model training complete!")

# Display learned hyperparameters
print("\nLearned GP Hyperparameters:")
print(f"  Noise variance: {gp_model.likelihood.noise.item():.6f}")
# Check if covar_module has outputscale (ScaleKernel) or is base kernel directly
if hasattr(gp_model.covar_module, 'outputscale'):
    print(f"  Output scale: {gp_model.covar_module.outputscale.item():.6f}")
    print(f"  Length scales: {gp_model.covar_module.base_kernel.lengthscale.detach().numpy()}")
else:
    # Direct access to kernel lengthscale
    print(f"  Length scales: {gp_model.covar_module.lengthscale.detach().numpy()}")

### Step 5: Optimize Acquisition Function

In [ ]:
# Create Expected Improvement acquisition function
best_f = y_train.max().item()
print(f"Best observed value: {best_f:.6f}")

EI = ExpectedImprovement(gp_model, best_f=best_f)

# Optimize acquisition function
print("\nOptimizing acquisition function...")
candidate, acq_value = optimize_acqf(
    EI,
    bounds=BOUNDS,
    q=1,
    num_restarts=NUM_RESTARTS,
    raw_samples=RAW_SAMPLES,
)

next_point = candidate.detach().numpy()[0]
print("✓ Optimization complete!")
print(f"\nProposed next sample point:")
print(f"  X_next = {next_point}")
print(f"  Expected Improvement: {acq_value.item():.6f}")

### Step 6: Visualize Results

For 3D problems, we visualize using multiple 2D projections and 3D scatter plots.

In [ ]:
# Visualize data points and next candidate in 3D
fig = plt.figure(figsize=(14, 6))

# 3D plot with next point
ax1 = fig.add_subplot(121, projection='3d')
scatter = ax1.scatter(X_init[:, 0], X_init[:, 1], X_init[:, 2], c=y_init, s=120, cmap='viridis', edgecolors='black', linewidth=1)
ax1.scatter(X_init[best_idx, 0], X_init[best_idx, 1], X_init[best_idx, 2], s=400, c='yellow', marker='*', edgecolors='black', linewidth=2, label='Best observed')
ax1.scatter(next_point[0], next_point[1], next_point[2], s=400, c='red', marker='*', edgecolors='black', linewidth=2, label='Next point')
ax1.set_xlabel('Compound 1')
ax1.set_ylabel('Compound 2')
ax1.set_zlabel('Compound 3')
ax1.set_title('3D Space with Next Sample Point', fontweight='bold')
ax1.legend()

# 2D projections showing all three pair plots
ax2 = fig.add_subplot(122)
colors = plt.cm.viridis((y_init - y_init.min()) / (y_init.max() - y_init.min()))
for i in range(len(X_init)):
    ax2.scatter(X_init[i, 0], X_init[i, 1], c=[colors[i]], s=100, edgecolors='black', linewidth=1, alpha=0.6)
ax2.scatter(next_point[0], next_point[1], s=400, c='red', marker='*', edgecolors='black', linewidth=2 , label='Next point')
ax2.set_xlabel('Compound 1')
ax2.set_ylabel('Compound 2')
ax2.set_title('Projection: Compound 1 vs 2', fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

### Step 7: Track Optimization Progress

In [ ]:
# Track best value over iterations
best_observed = np.maximum.accumulate(y_init)

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(best_observed) + 1), best_observed, 'b-o', linewidth=2, markersize=8, label='Best observed value')
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Best Objective Value', fontsize=12)
plt.title('Optimization Progress - Drug Discovery', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"Starting best value: {y_init.max():.6f}")
print(f"After {len(y_init)} initial samples")
print(f"\nNext submission: {next_point}")

### Visualize Expected Improvement

For higher-dimensional spaces, we visualize 1D slices of the acquisition function.
Each plot shows how EI changes along one dimension while others are fixed at the proposed point.

In [ ]:
# 1D marginal plots of Expected Improvement
n_points = 100
n_dims = len(next_point)

fig, axes = plt.subplots(1, n_dims, figsize=(4*n_dims, 4))
if n_dims == 1:
    axes = [axes]

for dim in range(n_dims):
    # Create points varying along this dimension
    X_marginal = np.tile(next_point, (n_points, 1))
    X_marginal[:, dim] = np.linspace(0, 1.0, n_points)
    X_marginal_torch = torch.tensor(X_marginal, dtype=torch.float64)
    
    # Compute EI at each point
    with torch.no_grad():
        ei_values = EI(X_marginal_torch.unsqueeze(1)).numpy()
    
    # Plot
    axes[dim].plot(X_marginal[:, dim], ei_values, 'b-', linewidth=2)
    axes[dim].axvline(next_point[dim], color='r', linestyle='--', linewidth=2, label='Proposed')
    axes[dim].set_xlabel(f'x{dim+1}', fontsize=12)
    axes[dim].set_ylabel('Expected Improvement' if dim == 0 else '', fontsize=12)
    axes[dim].set_title(f'EI along dim {dim+1}', fontsize=11, fontweight='bold')
    axes[dim].grid(True, alpha=0.3)
    if dim == 0:
        axes[dim].legend()

plt.suptitle('Expected Improvement - 1D Marginals', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Red dashed lines show the proposed next point coordinates.")
print(f"EI is maximized when considering all dimensions jointly.")

### Step 8: Format Next Query for Submission

Format the proposed next sample point in the required submission format:
- Format: `x1-x2-x3-...-xn` where each xᵢ begins with 0
- Precision: 6 decimal places per coordinate
- Range: All values clamped to [0, 1.0]

In [ ]:
# Format the next query for submission
def format_query(point):
    """Format point as x1-x2-...-xn with 6 decimal places, clamped to [0, 1.0]."""
    clamped = [max(0.0, min(1.0, x)) for x in point]
    return '-'.join([f'{x:.6f}' for x in clamped])

# Clamp next_point to valid range
next_point_clamped = np.array([max(0.0, min(1.0, x)) for x in next_point])

# Display the formatted submission query
submission_query = format_query(next_point)
print("=" * 60)
print("SUBMISSION QUERY FOR FUNCTION 3")
print("=" * 60)
print(f"\n{submission_query}\n")
print("=" * 60)
print(f"\nCoordinates breakdown:")
for i, x in enumerate(next_point, 1):
    print(f"  x{i} = {x:.6f}")
print(f"\nEI value: {acq_value.item():.6f}")
if acq_value.item() > 0.1:
    print("  -> High EI: Strong potential for improvement")
elif acq_value.item() > 0.001:
    print("  -> Moderate EI: Some exploration potential remains")
else:
    print("  -> Low EI: Approaching convergence")
if acq_value.item() > 0.1:
    print("  → High EI: Strong potential for improvement")
elif acq_value.item() > 0.001:
    print("  → Moderate EI: Some exploration potential remains")
else:
    print("  → Low EI: Approaching convergence")
print(f"Current best observed: {y_train.max().item():.6f}")

### Summary

**Initial Submission Complete - 3D Drug Discovery**

- Loaded 15 initial data points (3 compound combinations)
- Built GP surrogate model for 3D space
- Optimized Expected Improvement to find next compound combination
- Proposed next sample to minimize adverse reactions (maximize objective)
- Visualized using 3D scatter plots and 2D projections

**Next Steps:**
- Submit proposed compound combination
- Receive adverse reaction measurement
- Update dataset and retrain model

## Week 5 — Random Forest Surrogate

This section replaces the Gaussian Process surrogate with a **Random Forest** model for the drug discovery problem (f3).

**Why Random Forest for f3?**
- Handles noise well — individual trees may overfit, but the ensemble averages out noise
- No distributional assumptions about the objective function (unlike GP which assumes a Gaussian prior)
- Provides natural uncertainty estimates via variance across individual tree predictions
- Feature importance scores show which input dimensions matter most

**Acquisition Strategy:** Upper Confidence Bound (UCB) using tree-variance as uncertainty estimate.

### Step 1: Load Week 5 Data

Load the cumulative Week 5 data (20 total samples = initial + weekly submissions).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

# Load Week 5 cumulative data
X_w5 = np.load('../../data/f3/updated_inputs - Week 5.npy')
y_w5 = np.load('../../data/f3/updated_outputs - Week 5.npy')

print(f"Week 5 Data: {X_w5.shape[0]} samples, {X_w5.shape[1]} dimensions")
print(f"Input range:  [{X_w5.min():.6f}, {X_w5.max():.6f}]")
print(f"Output range: [{y_w5.min():.6f}, {y_w5.max():.6f}]")
print(f"Best observed value: {y_w5.max():.6f} at index {y_w5.argmax()}")
print(f"Best observed point: {X_w5[y_w5.argmax()]}")

### Step 2: Random Forest Hyperparameters

**Hyperparameter Choices and Justifications:**

1. **n_estimators = 150**: Number of trees in the forest. 150 trees for 3D data with 20 samples — slightly more trees to better capture the 3D landscape.

2. **max_depth = 6**: Maximum tree depth. Capped at 6 for 20 samples in 3D — allows slightly deeper trees for the extra dimension while keeping leaves populated.

3. **min_samples_split = 3**: Minimum samples required to split a node. Set to 3 to ensure no split creates nodes with too few samples.

4. **min_samples_leaf = 2**: Minimum samples in a leaf node. Set to 2 to prevent extreme predictions from single-sample leaves.

5. **random_state = 42**: Fixed seed for reproducibility.

6. **oob_score = True**: Out-of-bag score provides a free validation metric — each tree's OOB samples approximate hold-out error.

7. **UCB kappa = 2.0**: Exploration parameter for tree-variance-based uncertainty. Tree std is well-calibrated for this kappa value — provides good exploration while respecting the model's predictions.

8. **n_candidates = 20,000**: Random candidate points for UCB evaluation.

In [ ]:
# --- Random Forest Hyperparameters ---
N_ESTIMATORS = 150
MAX_DEPTH = 6
MIN_SAMPLES_SPLIT = 3
MIN_SAMPLES_LEAF = 2
RANDOM_STATE = 42
KAPPA = 2.0
N_CANDIDATES = 20000

print("Random Forest Surrogate Hyperparameters:")
print(f"  n_estimators:      {N_ESTIMATORS}")
print(f"  max_depth:         {MAX_DEPTH}")
print(f"  min_samples_split: {MIN_SAMPLES_SPLIT}")
print(f"  min_samples_leaf:  {MIN_SAMPLES_LEAF}")
print(f"  random_state:      {RANDOM_STATE}")
print(f"  oob_score:         True")
print(f"  UCB kappa:         {KAPPA}")
print(f"  UCB candidates:    {N_CANDIDATES}")

### Step 3: Train Random Forest Model

Fit the Random Forest on all available data. Display the OOB score and feature importance per input dimension.

In [ ]:
# Train Random Forest
rf_model = RandomForestRegressor(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    min_samples_split=MIN_SAMPLES_SPLIT,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    random_state=RANDOM_STATE,
    oob_score=True
)
rf_model.fit(X_w5, y_w5)

# Display results
print("Random Forest Training Results:")
print(f"  OOB Score (R²):  {rf_model.oob_score_:.6f}")
print(f"  Number of trees: {rf_model.n_estimators}")
print()
print("Feature Importance:")
for i, imp in enumerate(rf_model.feature_importances_):
    print(f"  x{i+1}: {imp:.4f} ({'*' * int(imp * 20)})")

### Step 4: UCB Acquisition Function

Compute the Upper Confidence Bound using:
- **mu(x)** = mean prediction across all trees (ensemble mean)
- **sigma(x)** = standard deviation of individual tree predictions (tree variance)
- **UCB(x) = mu(x) + kappa * sigma(x)** where kappa = 2.0

In [ ]:
# Generate random candidate points
np.random.seed(42)
candidates = np.random.uniform(0, 1.0, size=(N_CANDIDATES, 3))

# Get mean prediction from the ensemble
mu = rf_model.predict(candidates)

# Get uncertainty: std across individual tree predictions
tree_predictions = np.array([tree.predict(candidates) for tree in rf_model.estimators_])
sigma = tree_predictions.std(axis=0)

# UCB acquisition
ucb = mu + KAPPA * sigma

# Select best candidate
best_idx = np.argmax(ucb)
next_point_w5 = np.clip(candidates[best_idx], 0.0, 1.0)

print("UCB Acquisition Results:")
print(f"  Best UCB value:     {ucb[best_idx]:.6f}")
print(f"  Mean prediction:    {mu[best_idx]:.6f}")
print(f"  Tree std (sigma):   {sigma[best_idx]:.6f}")
print(f"  Next sample point:  {next_point_w5}")

### Step 5: Visualize Random Forest Surrogate

2D slice plot fixing the least important dimension at the best observed value.

In [ ]:
# Determine least important dimension and fix it at best observed value
importances = rf_model.feature_importances_
least_imp_dim = np.argmin(importances)
top_dims = [d for d in range(3) if d != least_imp_dim]
best_point = X_w5[y_w5.argmax()]
fixed_value = best_point[least_imp_dim]

print(f"Least important dimension: x{least_imp_dim+1} (importance={importances[least_imp_dim]:.4f})")
print(f"Fixed at best observed value: {fixed_value:.6f}")
print(f"Plotting dimensions: x{top_dims[0]+1} vs x{top_dims[1]+1}")

# Create 2D grid for the two most important dimensions
n_grid = 50
d0_grid = np.linspace(0, 1, n_grid)
d1_grid = np.linspace(0, 1, n_grid)
D0, D1 = np.meshgrid(d0_grid, d1_grid)

# Build full 3D grid points with least important dimension fixed
grid_points = np.zeros((n_grid * n_grid, 3))
grid_points[:, top_dims[0]] = D0.ravel()
grid_points[:, top_dims[1]] = D1.ravel()
grid_points[:, least_imp_dim] = fixed_value

# RF predictions and uncertainty on grid
grid_mu = rf_model.predict(grid_points).reshape(n_grid, n_grid)
grid_tree_preds = np.array([tree.predict(grid_points) for tree in rf_model.estimators_])
grid_sigma = grid_tree_preds.std(axis=0).reshape(n_grid, n_grid)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: RF mean prediction slice
ax1 = axes[0]
c1 = ax1.contourf(D0, D1, grid_mu, levels=30, cmap='viridis')
ax1.scatter(X_w5[:, top_dims[0]], X_w5[:, top_dims[1]], c='red', edgecolors='white', s=60, zorder=5, label='Observed')
ax1.scatter(next_point_w5[top_dims[0]], next_point_w5[top_dims[1]], c='yellow', marker='*', s=200, edgecolors='black', zorder=6, label='Next point')
ax1.set_xlabel(f'x{top_dims[0]+1}'); ax1.set_ylabel(f'x{top_dims[1]+1}')
ax1.set_title(f'RF Mean (x{least_imp_dim+1}={fixed_value:.3f})')
ax1.legend(loc='lower right', fontsize=8)
plt.colorbar(c1, ax=ax1)

# Plot 2: Uncertainty slice
ax2 = axes[1]
c2 = ax2.contourf(D0, D1, grid_sigma, levels=30, cmap='YlOrRd')
ax2.scatter(X_w5[:, top_dims[0]], X_w5[:, top_dims[1]], c='blue', edgecolors='white', s=60, zorder=5, label='Observed')
ax2.set_xlabel(f'x{top_dims[0]+1}'); ax2.set_ylabel(f'x{top_dims[1]+1}')
ax2.set_title(f'Uncertainty (x{least_imp_dim+1}={fixed_value:.3f})')
ax2.legend(loc='lower right', fontsize=8)
plt.colorbar(c2, ax=ax2)

# Plot 3: Feature importance
ax3 = axes[2]
dims = [f'x{i+1}' for i in range(3)]
colors = ['steelblue' if i != least_imp_dim else 'lightgray' for i in range(3)]
ax3.barh(dims, rf_model.feature_importances_, color=colors)
ax3.set_xlabel('Importance')
ax3.set_title('Feature Importance')
ax3.set_xlim(0, 1)

plt.tight_layout()
plt.show()

### Step 6: Convergence Plot

Running maximum (best observed value) across all observations.

In [ ]:
# Convergence plot
running_max = np.maximum.accumulate(y_w5)

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(y_w5) + 1), running_max, 'b-o', markersize=6, label='Running Maximum')
plt.scatter(range(1, len(y_w5) + 1), y_w5, c='gray', alpha=0.5, s=30, label='Individual Observations')
plt.axvline(x=15.5, color='red', linestyle='--', alpha=0.5, label='Initial -> Weekly boundary')
plt.xlabel('Observation Number')
plt.ylabel('Objective Value')
plt.title('Function 3 - Convergence Plot (Week 5)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best observed value: {y_w5.max():.6f}")
print(f"Achieved at observation: {y_w5.argmax() + 1}")

### Step 7: Format Submission Query

Format the proposed next sample point as `x1-x2-x3` with 6 decimal places.

In [ ]:
# Format submission query
def format_query(point):
    clamped = [max(0.0, min(1.0, x)) for x in point]
    return '-'.join([f'{x:.6f}' for x in clamped])

submission_query_w5 = format_query(next_point_w5)

print("=" * 60)
print("WEEK 5 SUBMISSION QUERY FOR FUNCTION 3")
print("=" * 60)
print(f"Surrogate: Random Forest ({N_ESTIMATORS} trees, max_depth={MAX_DEPTH})")
print(f"Acquisition: UCB (kappa={KAPPA})")
print(f"OOB Score: {rf_model.oob_score_:.6f}")
print(f"Next point: {next_point_w5}")
print(f"RF prediction: {mu[best_idx]:.6f}")
print(f"Tree uncertainty: {sigma[best_idx]:.6f}")
print(f"")
print(f">>> SUBMISSION: {submission_query_w5}")
print("=" * 60)

### Model Comparison

**Random Forest vs GP (Initial Section):**
- The GP model uses a Matern 5/2 kernel with Expected Improvement — assumes smooth, continuous functions with calibrated posterior uncertainty.
- The Random Forest makes no smoothness assumptions and builds predictions from ensemble of decision trees. Uncertainty from tree disagreement.
- For f3's drug discovery problem (3D), RF handles the extra dimension by automatically learning which dimensions matter most (via feature importance), similar to how GP lengthscales capture dimension relevance.
- Key trade-off: GP provides better-calibrated uncertainty, but RF is more robust to non-stationarity and discontinuities.

## Week 6 — Random Forest Surrogate

This section continues the Random Forest surrogate from Week 5 with a **reduced exploration parameter** (κ decreased from 2.0 to 0.5) to focus on exploitation.

**Why decrease κ for f3?**
- Week 5's Random Forest has learned the 3D landscape from 21 data points
- With reasonable OOB performance, we shift to exploitation — proposing points near the predicted optimum
- κ=0.5 makes the mean prediction dominate UCB, with tree variance providing only a small exploration bonus
- This consistent exploitation strategy (κ=0.5 for all f2–f8) enables fair comparison across surrogates

**Acquisition Strategy:** UCB with κ=0.5 (exploitation) using tree-variance as uncertainty estimate.

### Step 1: Load Week 6 Data

Load the cumulative Week 6 data (21 total samples = initial 15 + 6 weekly submissions).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

# Load Week 6 cumulative data
X_w6 = np.load('../../data/f3/updated_inputs - Week 6.npy')
y_w6 = np.load('../../data/f3/updated_outputs - Week 6.npy')

print(f"Week 6 Data: {X_w6.shape[0]} samples, {X_w6.shape[1]} dimensions")
print(f"Input range:  [{X_w6.min():.6f}, {X_w6.max():.6f}]")
print(f"Output range: [{y_w6.min():.6f}, {y_w6.max():.6f}]")
print(f"Best observed value: {y_w6.max():.6f} at index {y_w6.argmax()}")
print(f"Best observed point: {X_w6[y_w6.argmax()]}")

### Step 2: Random Forest Hyperparameters

**Hyperparameter Choices and Justifications:**

1. **n_estimators = 150**: Same as Week 5. 150 trees for 3D data.
2. **max_depth = 6**: Same as Week 5. Allows deeper trees for 3D landscape.
3. **min_samples_split = 3**, **min_samples_leaf = 2**: Same as Week 5.
4. **random_state = 42**: Fixed seed for reproducibility.
5. **oob_score = True**: Out-of-bag validation.
6. **UCB kappa = 0.5**: **Reduced from Week 5's κ=2.0** — exploitation focus.
7. **n_candidates = 20,000**: Same as Week 5.

In [ ]:
# --- Random Forest Hyperparameters ---
N_ESTIMATORS = 150
MAX_DEPTH = 6
MIN_SAMPLES_SPLIT = 3
MIN_SAMPLES_LEAF = 2
RANDOM_STATE = 42
KAPPA = 0.5              # Exploitation focus (Week 5: 2.0 → Week 6: 0.5)
N_CANDIDATES = 20000

print("Random Forest Surrogate Hyperparameters:")
print(f"  n_estimators:      {N_ESTIMATORS}")
print(f"  max_depth:         {MAX_DEPTH}")
print(f"  min_samples_split: {MIN_SAMPLES_SPLIT}")
print(f"  min_samples_leaf:  {MIN_SAMPLES_LEAF}")
print(f"  random_state:      {RANDOM_STATE}")
print(f"  oob_score:         True")
print(f"  UCB kappa:         {KAPPA} (Week 5: 2.0 → Week 6: 0.5)")
print(f"  UCB candidates:    {N_CANDIDATES}")

### Step 3: Train Random Forest Model

Fit the Random Forest on all available Week 6 data. Display the OOB score and feature importance per input dimension.

In [ ]:
# Train Random Forest
rf_model = RandomForestRegressor(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    min_samples_split=MIN_SAMPLES_SPLIT,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    random_state=RANDOM_STATE,
    oob_score=True
)
rf_model.fit(X_w6, y_w6)

# Display results
print("Random Forest Training Results:")
print(f"  OOB Score (R²):  {rf_model.oob_score_:.6f}")
print(f"  Number of trees: {rf_model.n_estimators}")
print()
print("Feature Importance:")
for i, imp in enumerate(rf_model.feature_importances_):
    print(f"  x{i+1}: {imp:.4f} ({'*' * int(imp * 20)})")

### Step 4: UCB Acquisition Function

Compute the Upper Confidence Bound using:
- **μ(x)** = mean prediction across all trees (ensemble mean)
- **σ(x)** = standard deviation of individual tree predictions (tree variance)
- **UCB(x) = μ(x) + κ · σ(x)** where κ = 0.5 (exploitation focus)

In [ ]:
# Generate random candidate points
np.random.seed(42)
candidates = np.random.uniform(0, 1.0, size=(N_CANDIDATES, 3))

# Get mean prediction from the ensemble
mu = rf_model.predict(candidates)

# Get uncertainty: std across individual tree predictions
tree_predictions = np.array([tree.predict(candidates) for tree in rf_model.estimators_])
sigma = tree_predictions.std(axis=0)

# UCB acquisition
ucb = mu + KAPPA * sigma

# Select best candidate
best_idx = np.argmax(ucb)
next_point_w6 = np.clip(candidates[best_idx], 0.0, 1.0)

print("UCB Acquisition Results:")
print(f"  Best UCB value:     {ucb[best_idx]:.6f}")
print(f"  Mean prediction:    {mu[best_idx]:.6f}")
print(f"  Tree std (sigma):   {sigma[best_idx]:.6f}")
print(f"  Exploitation contribution (μ): {mu[best_idx]:.6f}")
print(f"  Exploration contribution (κ·σ): {KAPPA * sigma[best_idx]:.6f}")
print(f"  Next sample point:  {next_point_w6}")

### Step 5: Visualize Random Forest Surrogate

2D slice plot fixing the least important dimension at the best observed value, with feature importance.

In [ ]:
# Determine least important dimension and fix it at best observed value
importances = rf_model.feature_importances_
least_imp_dim = np.argmin(importances)
top_dims = [d for d in range(3) if d != least_imp_dim]
best_point = X_w6[y_w6.argmax()]
fixed_value = best_point[least_imp_dim]

print(f"Least important dimension: x{least_imp_dim+1} (importance={importances[least_imp_dim]:.4f})")
print(f"Fixed at best observed value: {fixed_value:.6f}")
print(f"Plotting dimensions: x{top_dims[0]+1} vs x{top_dims[1]+1}")

# Create 2D grid for the two most important dimensions
n_grid = 50
d0_grid = np.linspace(0, 1, n_grid)
d1_grid = np.linspace(0, 1, n_grid)
D0, D1 = np.meshgrid(d0_grid, d1_grid)

# Build full 3D grid points with least important dimension fixed
grid_points = np.zeros((n_grid * n_grid, 3))
grid_points[:, top_dims[0]] = D0.ravel()
grid_points[:, top_dims[1]] = D1.ravel()
grid_points[:, least_imp_dim] = fixed_value

# RF predictions and uncertainty on grid
grid_mu = rf_model.predict(grid_points).reshape(n_grid, n_grid)
grid_tree_preds = np.array([tree.predict(grid_points) for tree in rf_model.estimators_])
grid_sigma = grid_tree_preds.std(axis=0).reshape(n_grid, n_grid)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: RF mean prediction slice
ax1 = axes[0]
c1 = ax1.contourf(D0, D1, grid_mu, levels=30, cmap='viridis')
ax1.scatter(X_w6[:, top_dims[0]], X_w6[:, top_dims[1]], c='red', edgecolors='white', s=60, zorder=5, label='Observed')
ax1.scatter(next_point_w6[top_dims[0]], next_point_w6[top_dims[1]], c='yellow', marker='*', s=200, edgecolors='black', zorder=6, label='Next point')
ax1.set_xlabel(f'x{top_dims[0]+1}'); ax1.set_ylabel(f'x{top_dims[1]+1}')
ax1.set_title(f'RF Mean (x{least_imp_dim+1}={fixed_value:.3f})')
ax1.legend(loc='lower right', fontsize=8)
plt.colorbar(c1, ax=ax1)

# Plot 2: Uncertainty slice
ax2 = axes[1]
c2 = ax2.contourf(D0, D1, grid_sigma, levels=30, cmap='YlOrRd')
ax2.scatter(X_w6[:, top_dims[0]], X_w6[:, top_dims[1]], c='blue', edgecolors='white', s=60, zorder=5, label='Observed')
ax2.set_xlabel(f'x{top_dims[0]+1}'); ax2.set_ylabel(f'x{top_dims[1]+1}')
ax2.set_title(f'Uncertainty (x{least_imp_dim+1}={fixed_value:.3f})')
ax2.legend(loc='lower right', fontsize=8)
plt.colorbar(c2, ax=ax2)

# Plot 3: Feature importance
ax3 = axes[2]
dims = [f'x{i+1}' for i in range(3)]
colors = ['steelblue' if i != least_imp_dim else 'lightgray' for i in range(3)]
ax3.barh(dims, rf_model.feature_importances_, color=colors)
ax3.set_xlabel('Importance')
ax3.set_title('Feature Importance')
ax3.set_xlim(0, 1)

plt.tight_layout()
plt.show()

### Step 6: Convergence Plot

Running maximum (best observed value) across all 21 observations.

In [ ]:
# Convergence plot
running_max = np.maximum.accumulate(y_w6)

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(y_w6) + 1), running_max, 'b-o', markersize=6, label='Running Maximum')
plt.scatter(range(1, len(y_w6) + 1), y_w6, c='gray', alpha=0.5, s=30, label='Individual Observations')
plt.axvline(x=15.5, color='red', linestyle='--', alpha=0.5, label='Initial → Weekly boundary')
plt.xlabel('Observation Number')
plt.ylabel('Objective Value')
plt.title('Function 3 — Convergence Plot (Week 6)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best observed value: {y_w6.max():.6f}")
print(f"Achieved at observation: {y_w6.argmax() + 1}")

### Step 7: Format Submission Query

Format the proposed next sample point as `x1-x2-x3` with 6 decimal places, clamped to [0.0, 1.0].

In [ ]:
# Format submission query
def format_query(point):
    """Format point as x1-x2-...-xn with 6 decimal places, clamped to [0.0, 1.0]."""
    clamped = [max(0.0, min(1.0, x)) for x in point]
    return '-'.join([f'{x:.6f}' for x in clamped])

submission_query_w6 = format_query(next_point_w6)

print("=" * 60)
print("WEEK 6 SUBMISSION QUERY FOR FUNCTION 3")
print("=" * 60)
print(f"Surrogate: Random Forest ({N_ESTIMATORS} trees, max_depth={MAX_DEPTH})")
print(f"Acquisition: UCB (κ={KAPPA})")
print(f"Strategy: EXPLOITATION (κ reduced from 2.0 to 0.5)")
print(f"OOB Score: {rf_model.oob_score_:.6f}")
print(f"Next point: {next_point_w6}")
print(f"RF prediction: {mu[best_idx]:.6f}")
print(f"Tree uncertainty: {sigma[best_idx]:.6f}")
print(f"\n>>> SUBMISSION: {submission_query_w6}")
print("=" * 60)

# Research
Below is a **Bayesian Optimization playbook** tailored to your NeurIPS‑style drug‑discovery setup:

*   **Inputs**: 3 continuous decision variables (dose levels of compounds A, B, C) per experiment (each row of your 3D array).
*   **Output**: count of adverse reactions (1D array); you’ll **maximize** a transformed objective (e.g., the negative or a variance‑stabilized transform of side‑effect counts).
*   **Challenges**: noisy responses, potential **interaction (synergy/antagonism)** among compounds, and **multiple local optima**.

***

## 1) Surrogate model

### A. If the doses are **unconstrained box‑bounded** (each in $$[L_i,U_i]$$)

Use a **Gaussian Process (GP)** with a **Matérn‑5/2 kernel** and **ARD** lengthscales.

*   **Why**: 3D is GP‑friendly; Matérn‑5/2 balances smoothness with flexibility; ARD adapts per‑compound sensitivity.
*   **Noise**: counts imply variance grows with the mean; either (i) apply a **variance‑stabilizing transform** (see §3), then model with **Gaussian noise**, or (ii) use a **Student‑t likelihood** for robustness.

**Suggested GP configuration**

```text
Mean:        Constant (learned)
Kernel:      Matérn-5/2 with ARD (ℓA, ℓB, ℓC)
Likelihood:  Gaussian noise  (or Student-t if heavy-tail outliers)
Scaling:     Standardize inputs to [0,1]; standardize transformed outputs (z-score)
Fit:         Maximize marginal log-likelihood with 10–20 random restarts
```

### B. If doses are **compositional** (e.g., non-negative and sum to a fixed total)

Work on the **simplex**. Apply a **log‑ratio transform** (e.g., **additive log‑ratio, ALR**, or **isometric log‑ratio, ILR**) to map the 3‑component composition to a 2‑D unconstrained coordinate, then use the GP as above **in the transformed space**. This respects mixture geometry and drastically reduces artifacts common with raw‑space GPs.

> If you prefer tree‑based robustness over perfect geometry, **BART** (Bayesian Additive Regression Trees) is an excellent surrogate in 3D, particularly when responses are rugged or contain outliers; uncertainty comes from the posterior over trees and works well with UCB/NEI.

***

## 2) Acquisition function

### Primary: **Noisy Expected Improvement (NEI)** (or **qNEI** for batches)

*   **Why**: explicitly accounts for noise; reliable under multi‑modal objectives.
*   **Batching**: if you run parallel assays, use **qNEI** with local penalization or diversity encouragement so proposed points spread across the design space.

### Warm‑start (early iterations): **UCB** (Upper Confidence Bound)

*   Use a **high exploration weight** initially to ensure broad coverage, then switch to NEI for stable, noise‑aware progress.

> If you’re budget‑rich and want maximum one‑step value, **Knowledge Gradient (KG)** is excellent under noise—more compute‑intensive but very sample‑efficient.

***

## 3) Objective transformation (critical for counts)

Since you measure **counts of adverse reactions**, the noise is inherently **heteroscedastic** (Poisson‑like). Two practical routes:

1.  **Variance‑stabilize then use Gaussian likelihood**
    *   **Anscombe transform**: $$y' = 2\sqrt{y + \tfrac{3}{8}}$$
    *   Or **log1p**: $$y' = \log(1 + y)$$
    *   Then **maximize** $$f(x) = -\,y'$$ (equivalently, minimize $$y'$$)

2.  **Model the count directly**
    *   GP with **Poisson/Negative Binomial likelihood** (heavier setup; acquisition derivations can be trickier)
    *   For simplicity + strong performance, most BO stacks use route (1) with NEI.

***

## 4) Hyperparameters & practical settings (ready to use)

### 4.1 GP priors / inits (inputs scaled to $$[0,1]^3$$)

```text
Lengthscales (ARD):    init ℓA=ℓB=ℓC = 0.25   (¼ domain)
  prior log ℓ ~ N(log 0.25, 1.0^2)
Signal variance σ_f^2: init to Var(y')   (with half-Cauchy or log-normal prior)
Noise variance  σ_n^2: init to 0.05·Var(y')  (half-Cauchy prior; infer jointly)
Jitter:                1e-6 to 1e-8
Optimization:          L-BFGS-B; 10–20 random restarts for the hyperparameters
```

### 4.2 Acquisition—NEI / qNEI

```text
NEI fantasies:     32 (increase to 64 if compute allows)
Batch size (q):    2–4 (if parallel experiments)
Diversity:         local penalization or batch diversity penalty
Acq optimizer:     3000–5000 Sobol seeds → keep top 100 → L-BFGS restarts
```

### 4.3 UCB warm‑start

```text
UCB(x) = μ(x) + κ_t·σ(x)
κ schedule:   start κ=3.0 for first ~15–20% of budget; anneal to κ≈0.7–1.0
```

### 4.4 Initialization, replication, and budget use

```text
Initial design:  16–24 Sobol points across [0,1]^3  (cover space well)
Replicates:      3–5 re-measurements at 2–4 selected points
                 (helps estimate noise, especially near current best)
Switch point:    after 2–3 UCB iterations, move to NEI/qNEI
Stopping:        no improvement ≥1% of running std for 10 iterations, or budget end
```

### 4.5 Simplex (composition) case—log‑ratio specifics

If A, B, C are non‑negative and $$A+B+C=\text{const}$$:

*   Use **ALR**: $$(u_1,u_2) = (\log(A/C), \log(B/C))$$, fit the GP in $$(u_1,u_2)$$.
*   When optimizing the acquisition, search in $$(u_1,u_2)$$ and map back to $$(A,B,C)$$ (enforce feasibility).
*   Hyperparameters as above but in 2D; often **ℓ ≈ 0.2–0.3** is a good starting point.

***

## 5) Guardrails for drug discovery

*   **Safety caps**: If any dose combination could be risky, run BO with **constraints** (e.g., keep expected adverse reactions below a threshold). Use **constrained NEI** (objective = −transformed side‑effects; constraint = $$P\{\text{side‑effects} \le \tau\}\ge 1-\alpha$$).
*   **Monotonicity (optional)**: If pharmacology suggests monotone trends (e.g., increasing a specific compound never reduces adverse reactions), consider a **monotone GP** or place a **virtual derivative prior**—only if you’re confident in the biology.
*   **Batch design**: Two to four points per batch is a sweet spot; larger batches reduce adaptivity.

***

## 6) Alternative surrogate (robustness‑first)

If preliminary runs show **rugged** surfaces or **outliers**:

*   Use **BART** (or **Random Forest**) on transformed $$y'$$ with **UCB → NEI**.
*   BART hyperparameters: \~200 trees, shallow depth prior (α=0.95, β=2), standardize inputs/outputs, run 2–4k posterior draws for stable uncertainty.

***

## 7) A minimal, copy‑paste configuration

```text
Goal: minimize side-effects; maximize f(x) = - Anscombe(y) where y is adverse count
Transform: y' = 2*sqrt(y + 3/8); f(x) = -y'

Surrogate: GP (Matérn-5/2, ARD), Gaussian noise
Inputs:    scaled to [0,1]^3  (or ALR transform if on simplex)

Init:      20 Sobol points; replicate 3 of them 3× each
Warm-up:   UCB with κ=3 → 2 over first 3 iterations
Main BO:   qNEI (q=3), fantasies=32, local penalization
Acq opt:   4000 Sobol starts → 100 L-BFGS restarts per iteration
Fit:       MLL with 15 random restarts; jitter=1e-6

Stop:      budget exhausted OR no NEI gain ≥1% of rolling std for 10 iters
```

***

### One quick clarification (optional but helpful)

Are your three compound amounts **constrained to a fixed total (mixture/simplex)**, or are they **independent within bounds**?  
If it’s a simplex, I’ll tailor the exact ALR/ILR mapping and acquisition search box for you.
